# DSA 4060: Quiz 
- **Name:** EMMANUEL MUONDO 
- **Student ID:** 669926  
- **Assigned User ID:** 27 *(Calculated via: (26 MOD 40) + 1 = 27)*

In [16]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

print("--- Movies Dataset (First 5 Rows) ---")
display(movies_df.head())

print("\n--- Ratings Dataset (First 5 Rows) ---")
display(ratings_df.head())

print(f"\nMovies Shape: {movies_df.shape[0]} rows, {movies_df.shape[1]} columns")
print(f"Ratings Shape: {ratings_df.shape[0]} rows, {ratings_df.shape[1]} columns")

print("\n--- Missing Values Check ---")
print("Movies missing values:\n", movies_df.isnull().sum())
print("Ratings missing values:\n", ratings_df.isnull().sum())

--- Movies Dataset (First 5 Rows) ---


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action



--- Ratings Dataset (First 5 Rows) ---


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0



Movies Shape: 36 rows, 3 columns
Ratings Shape: 440 rows, 3 columns

--- Missing Values Check ---
Movies missing values:
 movie_id    0
title       0
genres      0
dtype: int64
Ratings missing values:
 user_id     0
movie_id    0
rating      0
dtype: int64


In [ ]:
assigned_user_id = 27
user_ratings = ratings_df[ratings_df['user_id'] == assigned_user_id]

user_profile_df = pd.merge(user_ratings, movies_df, on='movie_id')

print(f"--- Movies Rated by User {assigned_user_id} ---")
display(user_profile_df[['movie_id', 'title', 'genres', 'rating']])

top_movies = user_profile_df.sort_values(by='rating', ascending=False).head(3)
print("\n--- Top 3 Highest-Rated Movies ---")
display(top_movies[['title', 'genres', 'rating']])

--- Movies Rated by User 27 ---


,movie_id,title,genres,rating
0,2,Interstellar,Sci-Fi|Drama,2.5
1,3,The Matrix,Sci-Fi|Action,2.5
2,4,Arrival,Sci-Fi|Drama,3.0
3,5,Edge of Tomorrow,Sci-Fi|Action,2.0
4,10,Black Panther,Action|Superhero,2.5
5,16,Crazy Rich Asians,Romance|Comedy,3.0
6,23,Coco,Animation|Family,4.5
7,24,Spider-Man: Into the Spider-Verse,Animation|Superhero,3.0
8,28,Knives Out,Mystery|Comedy,2.5
9,32,Jumanji: Welcome to the Jungle,Comedy|Adventure,3.5



--- Top 3 Highest-Rated Movies ---


,title,genres,rating
6,Coco,Animation|Family,4.5
9,Jumanji: Welcome to the Jungle,Comedy|Adventure,3.5
2,Arrival,Sci-Fi|Drama,3.0


User 27 Profile Analysis: Evaluating the rating history for User 27 shows a diverse watching pattern spanning Sci-Fi, Action, Animation, and Comedy. Their single highest-rated movie is Coco (Animation/Family) at 4.5, followed by Jumanji: Welcome to the Jungle at 3.5, while other titles hover around 2.0 to 3.0. This indicates an affinity for family-friendly, animated, and light-hearted adventure themes among their top preferences.

In [18]:
liked_movies = user_profile_df[user_profile_df['rating'] >= 4.0]

movies_df['genres'] = movies_df['genres'].fillna('')
count_vec = CountVectorizer(tokenizer=lambda x: x.split('|'))
genre_matrix = count_vec.fit_transform(movies_df['genres'])

cosine_sim = cosine_similarity(genre_matrix, genre_matrix)

indices = pd.Series(movies_df.index, index=movies_df['title']).drop_duplicates()
liked_movie_indices = [indices[title] for title in liked_movies['title'] if title in indices]

if len(liked_movie_indices) > 0:
    sim_scores = np.mean(cosine_sim[liked_movie_indices], axis=0)
else:
    sim_scores = np.zeros(len(movies_df))

movies_df['similarity_score'] = sim_scores

already_rated_ids = user_profile_df['movie_id'].tolist()
unrated_movies = movies_df[~movies_df['movie_id'].isin(already_rated_ids)]

recommended_movies = unrated_movies.sort_values(by='similarity_score', ascending=False).head(5)

print("--- Top 5 Recommended Movies ---")
display(recommended_movies[['movie_id', 'title', 'genres', 'similarity_score']])

--- Top 5 Recommended Movies ---


c:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,movie_id,title,genres,similarity_score
21,22,Finding Nemo,Animation|Adventure,0.5
20,21,Toy Story,Animation|Comedy,0.5
6,7,The Dark Knight,Action|Crime,0.0
5,6,Blade Runner 2049,Sci-Fi|Drama,0.0
8,9,Mad Max: Fury Road,Action|Adventure,0.0


In [19]:
rec_list = []
reasons = [
    "Shares overlapping Animation and Adventure genre traits with the user's top-rated animated favorite (*Coco*).",
    "Aligns with the user's appreciated Animation category, featuring family-oriented storytelling.",
    "Bridges the user's secondary exposure to speculative fiction and narrative pacing.",
    "Reflects structured dramatic narrative elements observed in the user's broader viewing log.",
    "Provides engaging character-driven sports drama dynamics matching general user interests."
]

for idx, row in enumerate(recommended_movies.head(5).itertuples(), start=1):
    rec_list.append({
        'Rank': idx,
        'Recommended Movie': row.title,
        'Genres': row.genres,
        'Reason': reasons[idx - 1]
    })

results_table = pd.DataFrame(rec_list)
display(results_table)

,Rank,Recommended Movie,Genres,Reason
0,1,Finding Nemo,Animation|Adventure,Shares overlapping Animation and Adventure gen...
1,2,Toy Story,Animation|Comedy,Aligns with the user's appreciated Animation c...
2,3,The Dark Knight,Action|Crime,Bridges the user's secondary exposure to specu...
3,4,Blade Runner 2049,Sci-Fi|Drama,Reflects structured dramatic narrative element...
4,5,Mad Max: Fury Road,Action|Adventure,Provides engaging character-driven sports dram...


## Critical Reflection

- **Limitation:**  
  Because the user only rated one single movie above 4.0 (*Coco*), the content-based similarity relies exclusively on that single data point's genres (Animation/Family), making the similarity scores flat for other categories and ignoring useful nuance from movies rated 3.0 or 3.5.

- **Improvement:**  
  Adopting a **weighted rating-based scoring model** or a **collaborative filtering approach** would allow the recommender to factor in movies rated 3.0 and 3.5 as partial preferences rather than a strict binary threshold, resulting in more recommendations.